In [1]:
# =====================================================================
#  XAI Framework – Imports & Configuration
# =====================================================================
import os
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Any

# Machine learning / SHAP
import shap
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
import lightgbm as lgb
try:
    import catboost
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

# for LLM groq
import requests 

warnings.filterwarnings('ignore')

# ------------------- Paths & Config -------------------
CONFIG = {
    "DATA_ROOT":        "./prepareddata",          # raw test/val + training CSVs
    "TRAINED_ROOT":     "./trained",               # saved model artifacts
    "OUTPUT_DIR":       "./trained/xai",           # where to save XAI outputs
    "LEADERBOARD_DIR":  "./trained/leaderboards",
    "TARGET_COL":       "Fraud",
    "RANDOM_STATE":     42,
    "GEMINI_API_KEY":   "AQ.Ab8RN6IWKaUx2cfxOtsjuAS2u7xUHpO7NShnbuvcSlhF1gNlGw",                        # Gemini API key here
}

Path(CONFIG["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)

# ------------------- Helper Functions (from training.ipynb) -------------------
def safe_columns(df):
    import re
    df = df.rename(columns=lambda c: re.sub(r'[^a-zA-Z0-9_]', '_', str(c)))
    return df

def _detect_types(X: pd.DataFrame):
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    int_cols = X.select_dtypes(include=['int']).columns
    for c in int_cols:
        if c not in cat_cols and X[c].nunique() < 10:
            cat_cols.append(c)
            num_cols.remove(c)
    return cat_cols, num_cols

def _add_engineered_features(df, dataset_name, agg_stats):
    df = df.copy()
    if dataset_name == "Sparkov":
        if "unix_time" in df.columns:
            trans_dt = pd.to_datetime(df["unix_time"], unit='s')
            df["hour"] = trans_dt.dt.hour
            df["dayofweek"] = trans_dt.dt.dayofweek
            df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
            df["month"] = trans_dt.dt.month
        if "amt" in df.columns:
            df["log_amt"] = np.log1p(df["amt"])
        if "cc_num" in df.columns and "sparkov_card_stats" in agg_stats:
            df = df.merge(agg_stats["sparkov_card_stats"], on="cc_num", how="left")
        if "merchant" in df.columns and "sparkov_merch_stats" in agg_stats:
            df = df.merge(agg_stats["sparkov_merch_stats"], on="merchant", how="left")
        if "merch_lat" in df.columns and "merch_long" in df.columns:
            def haversine_vectorised(lat1, lon1, lat2, lon2):
                lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
                dlat = lat2 - lat1
                dlon = lon2 - lon1
                a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
                c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
                return 6371.0 * c
            df["customer_merchant_dist"] = haversine_vectorised(
                df["lat"].values, df["long"].values,
                df["merch_lat"].values, df["merch_long"].values
            )
    elif dataset_name == "IEEE-CIS":
        if "TransactionDT" in df.columns:
            start_date = pd.Timestamp("2017-12-01")
            dt = start_date + pd.to_timedelta(df["TransactionDT"], unit='s')
            df["hour"] = dt.dt.hour
            df["dayofweek"] = dt.dt.dayofweek
            df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
            df["month"] = dt.dt.month
        if "TransactionAmt" in df.columns:
            df["log_TransactionAmt"] = np.log1p(df["TransactionAmt"])
        if "card1" in df.columns and "ieee_card1_stats" in agg_stats:
            df = df.merge(agg_stats["ieee_card1_stats"], on="card1", how="left")
        if "addr1" in df.columns and "ieee_addr1_stats" in agg_stats:
            df = df.merge(agg_stats["ieee_addr1_stats"], on="addr1", how="left")
        for col in ["P_emaildomain", "R_emaildomain"]:
            key = f"ieee_{col}_stats"
            if col in df.columns and key in agg_stats:
                df = df.merge(agg_stats[key], on=col, how="left")
    elif dataset_name == "EuropeanCard":
        if "Time" in df.columns:
            df["hour"] = (df["Time"] // 3600) % 24
            df["day"]  = df["Time"] // (24 * 3600)
            df["week"] = df["Time"] // (7 * 24 * 3600)
        if "Amount" in df.columns:
            df["log_Amount"] = np.log1p(df["Amount"])
    return df

def apply_preprocessing(raw_df, base_state, combo_state, dataset_name,
                        target_col="Fraud", training_columns=None):
    if target_col in raw_df.columns:
        y = raw_df[target_col].values
        X = raw_df.drop(columns=[target_col])
    else:
        y = None
        X = raw_df.copy()

    imp_cat = base_state["imputer_cat"]
    imp_num = base_state["imputer_num"]

    keep_cols = []
    if hasattr(imp_cat, 'feature_names_in_'):
        keep_cols.extend(imp_cat.feature_names_in_)
    if hasattr(imp_num, 'feature_names_in_'):
        keep_cols.extend(imp_num.feature_names_in_)
    X = X[[c for c in keep_cols if c in X.columns]].copy()

    cat_cols_imp = [c for c in imp_cat.feature_names_in_ if c in X.columns] if hasattr(imp_cat, 'feature_names_in_') else []
    num_cols_imp = [c for c in imp_num.feature_names_in_ if c in X.columns] if hasattr(imp_num, 'feature_names_in_') else []
    if cat_cols_imp:
        X[cat_cols_imp] = imp_cat.transform(X[cat_cols_imp])
    if num_cols_imp:
        X[num_cols_imp] = imp_num.transform(X[num_cols_imp])

    X = _add_engineered_features(X, dataset_name, base_state["agg_stats"])
    X = X.fillna(0)

    ID_DROP_COLS = ["cc_num", "merchant", "nameOrig", "nameDest", "trans_num",
                    "TransactionID", "card1", "addr1", "P_emaildomain", "R_emaildomain",
                    "DeviceInfo"]
    for col in ID_DROP_COLS:
        if col in X.columns:
            X.drop(columns=col, inplace=True)

    freq_maps = base_state["freq_maps"]
    for col, fmap in freq_maps.items():
        if col in X.columns:
            X[col + "_freq"] = X[col].map(fmap).fillna(0)
            X.drop(columns=col, inplace=True)

    cat_cols = base_state["cat_cols"]
    num_cols = base_state["num_cols"]
    all_final_cols = cat_cols + num_cols
    X = X.reindex(columns=all_final_cols, fill_value=0)

    if cat_cols:
        X[cat_cols] = base_state["encoder"].transform(X[cat_cols])
    if num_cols:
        X[num_cols] = base_state["scaler"].transform(X[num_cols])

    selector = combo_state.get("selector", None)
    if selector is not None:
        if hasattr(selector, 'feature_names_in_'):
            X = X.reindex(columns=list(selector.feature_names_in_), fill_value=0)
        mask = selector.get_support()
        X = X.loc[:, mask]

    if training_columns is not None:
        X = X.reindex(columns=training_columns, fill_value=0)

    return X

print("[config] XAI notebook ready.")

[config] XAI notebook ready.


In [2]:
# =====================================================================
#  Select Best Ensemble (Highest F1) for Each Dataset
# =====================================================================
import joblib

results_path = Path(CONFIG["OUTPUT_DIR"]).parent / "optimized_hybrid" / "optimized_hybrid_results.csv"
results_df = pd.read_csv(results_path)

best_ensembles = {}
for ds in results_df["dataset"].unique():
    sub = results_df[results_df["dataset"] == ds]
    if sub.empty:
        continue
    best_row = sub.loc[sub["f1"].idxmax()]
    best_ensembles[ds] = {
        "strategy": best_row["selection_strategy"],
        "meta_learner": best_row["meta_learner"],
        "n_models": best_row["n_models"],
        "f1": best_row["f1"],
    }
    print(f"[select] {ds}: best ensemble = {best_row['selection_strategy']} "
          f"(meta={best_row['meta_learner']}, F1={best_row['f1']:.4f})")

[select] Sparkov: best ensemble = diverse_cv (meta=RandomForest, F1=0.8252)
[select] EuropeanCard: best ensemble = balanced (meta=RidgeClassifier, F1=0.8550)
[select] IEEE-CIS: best ensemble = diverse_cv (meta=LightGBM, F1=0.4582)


In [3]:
# =====================================================================
#  Base Model Factory & Unsupervised Wrapper
# =====================================================================
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import GaussianNB, ComplementNB
from sklearn.neighbors import KNeighborsClassifier

class UnsupervisedAnomalyClassifier(BaseEstimator, ClassifierMixin):
    """
    Wraps IsolationForest, OneClassSVM, LocalOutlierFactor to provide
    predict_proba() for SHAP.
    """
    def __init__(self, detector):
        self.detector = detector

    def fit(self, X, y=None):
        self.detector_ = clone(self.detector)
        self.detector_.fit(X)
        train_scores = self._raw_score(X)
        self.score_min_ = np.min(train_scores)
        self.score_max_ = np.max(train_scores)
        self.classes_ = np.array([0, 1])
        return self

    def _raw_score(self, X):
        if hasattr(self.detector_, "decision_function"):
            return self.detector_.decision_function(X)
        elif hasattr(self.detector_, "score_samples"):
            return self.detector_.score_samples(X)
        else:
            return self.detector_.predict(X).astype(float)

    def predict_proba(self, X):
        raw = self._raw_score(X)
        eps = 1e-8
        norm = np.clip((raw - self.score_min_) / (self.score_max_ - self.score_min_ + eps), 0, 1)
        fraud_prob = 1.0 - norm
        legit_prob = norm
        return np.vstack([legit_prob, fraud_prob]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def get_base_model(model_name, random_state=CONFIG["RANDOM_STATE"]):
    """Return an instance of the base model with the same hyperparameters
    as in the original training."""
    if model_name == "lgb":
        return lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05,
            num_leaves=31, max_depth=-1,
            subsample=0.9, colsample_bytree=0.9,
            reg_alpha=0.0, reg_lambda=0.0,
            objective='binary', n_jobs=-1,
            random_state=random_state, verbose=-1,
        )
    elif model_name == "xgb":
        return XGBClassifier(
            n_estimators=300, learning_rate=0.05,
            max_depth=6, subsample=0.9, colsample_bytree=0.9,
            tree_method="hist", eval_metric="auc",
            n_jobs=-1, random_state=random_state,
            verbosity=0, use_label_encoder=False,
        )
    elif model_name == "cat" and HAS_CATBOOST:
        return CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=6,
            l2_leaf_reg=3.0, random_seed=random_state,
            verbose=0, allow_writing_files=False, thread_count=-1,
        )
    elif model_name == "rf":
        return RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=1,
            n_jobs=-1, random_state=random_state,
        )
    elif model_name == "logreg":
        return LogisticRegression(
            C=0.5, class_weight="balanced", max_iter=500,
            random_state=random_state
        )
    elif model_name == "linsvc":
        return LinearSVC(C=0.5, class_weight="balanced", max_iter=2000,
                         random_state=random_state)
    # ---- Additional models from original pool ----
    elif model_name == "gaussian_nb":
        return GaussianNB()
    elif model_name == "complement_nb":
        return Pipeline([
            ("scale", MinMaxScaler()),
            ("clf", ComplementNB())
        ])
    elif model_name == "sgd_log":
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", SGDClassifier(loss="log_loss", penalty="elasticnet",
                                  l1_ratio=0.15, alpha=1e-4,
                                  max_iter=50, tol=1e-3,
                                  random_state=random_state, n_jobs=-1))
        ])
    elif model_name == "sgd_huber":
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", SGDClassifier(loss="modified_huber", penalty="l2",
                                  alpha=1e-4, max_iter=50, tol=1e-3,
                                  random_state=random_state, n_jobs=-1))
        ])
    elif model_name == "knn":
        # n_neighbors depends on n_features; will set dynamically later if needed
        return KNeighborsClassifier(n_neighbors=5, n_jobs=-1, weights="distance")
    elif model_name == "isolation_forest":
        return UnsupervisedAnomalyClassifier(
            IsolationForest(n_estimators=200, contamination=0.1,
                            random_state=random_state, n_jobs=-1)
        )
    elif model_name == "one_class_svm":
        return UnsupervisedAnomalyClassifier(
            OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
        )
    elif model_name == "local_outlier_factor":
        return UnsupervisedAnomalyClassifier(
            LocalOutlierFactor(novelty=True, contamination=0.1, n_jobs=-1)
        )
    elif model_name.startswith("dl_"):
        # Deep learning models are too heavy to retrain for SHAP; raise to skip
        raise ValueError(f"Deep learning model '{model_name}' is not supported for SHAP retraining.")
    else:
        raise ValueError(f"Unsupported base model: {model_name}")

In [4]:
# =====================================================================
#  Load Preprocessing States for Each Dataset
# =====================================================================
base_states = {}
for ds in best_ensembles.keys():
    base_path = Path(CONFIG["DATA_ROOT"]) / f"{ds}_base_state.joblib"
    if not base_path.exists():
        raise FileNotFoundError(f"Missing base_state for {ds}: {base_path}")
    base_states[ds] = joblib.load(base_path)

# Load selected model details from JSON files
selected_models_info = {}
for ds, info in best_ensembles.items():
    strategy = info["strategy"]
    json_path = Path(CONFIG["OUTPUT_DIR"]).parent / "optimized_hybrid" / f"{ds}_selected_models_{strategy}.json"
    if json_path.exists():
        with open(json_path, "r") as f:
            selected_models_info[ds] = json.load(f)
        print(f"[load] {ds}: selected models from {json_path}")
    else:
        print(f"[warn] Missing selected models JSON for {ds}/{strategy}")

[load] Sparkov: selected models from trained/optimized_hybrid/Sparkov_selected_models_diverse_cv.json
[load] EuropeanCard: selected models from trained/optimized_hybrid/EuropeanCard_selected_models_balanced.json
[load] IEEE-CIS: selected models from trained/optimized_hybrid/IEEE-CIS_selected_models_diverse_cv.json


In [5]:
# =====================================================================
#  Level 1: Base Model SHAP Analysis
# =====================================================================
# For each dataset, retrain each selected base model on its resampled
# training CSV and compute mean |SHAP| on test features.

shap_base_results = {}   # ds -> {model_name: (model, shap_values_mean, feature_names, X_test)}

for ds, info in best_ensembles.items():
    print(f"\n{'='*70}\nDataset: {ds}\n{'='*70}")
    strategy = info["strategy"]
    selected = selected_models_info.get(ds, {}).get("selected_models", [])
    if not selected:
        print(f"No selected models for {ds}, skipping.")
        continue

    # Load raw test data
    raw_test = pd.read_csv(Path(CONFIG["DATA_ROOT"]) / f"{ds}_test.csv")
    y_test = raw_test[CONFIG["TARGET_COL"]].values

    shap_base_results[ds] = {}

    for sel in selected:
        variant = sel["variant"]
        model_name = sel["model"]
        print(f"\n--- Base model: {model_name} (variant: {variant}) ---")

        # Locate resampled training CSV
        train_csv = Path(CONFIG["DATA_ROOT"]) / f"{variant}.csv"
        if not train_csv.exists():
            print(f"  [skip] Training CSV not found: {train_csv}")
            continue

        train_df = pd.read_csv(train_csv)
        X_train_raw = train_df.drop(columns=[CONFIG["TARGET_COL"]])
        y_train = train_df[CONFIG["TARGET_COL"]].values

        # Load combo_state for this variant
        combo_path = Path(CONFIG["DATA_ROOT"]) / f"{variant}_state.joblib"
        if combo_path.exists():
            combo_state = joblib.load(combo_path)
        else:
            combo_state = {}

        # Apply preprocessing to raw test to get features in the same space
        X_test_raw = apply_preprocessing(
            raw_test, base_states[ds], combo_state, ds,
            target_col=CONFIG["TARGET_COL"],
            training_columns=X_train_raw.columns.tolist()
        )

        # ---- Sanitize column names (as done in original training) ----
        X_train = safe_columns(X_train_raw)
        X_test  = safe_columns(X_test_raw)

        # Ensure test has the same columns as train (after sanitization)
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        # Train base model (skip if unsupported)
        try:
            model = get_base_model(model_name)
            model.fit(X_train, y_train)
        except Exception as e:
            print(f"  [skip] Could not train model {model_name}: {e}")
            continue

        # Compute SHAP
        explainer = None
        if model_name in ["lgb", "xgb", "cat"] or isinstance(model, RandomForestClassifier):
            explainer = shap.TreeExplainer(model)
        elif isinstance(model, LogisticRegression) or isinstance(model, LinearSVC):
            explainer = shap.LinearExplainer(model, X_train)
        else:
            try:
                explainer = shap.KernelExplainer(model.predict_proba, shap.sample(X_train, 50))
            except:
                print(f"  [skip] SHAP not supported for {model_name}")
                continue

        shap_values = explainer.shap_values(X_test)
        if isinstance(shap_values, list):
            # For binary classification, take positive class
            shap_values = shap_values[1]
        # shap_values shape: (n_samples, n_features)
        mean_abs_shap = np.abs(shap_values).mean(axis=0)

        shap_base_results[ds][model_name] = {
            "model": model,
            "shap_values": shap_values,
            "mean_abs_shap": mean_abs_shap,
            "feature_names": X_train.columns.tolist(),
            "X_test": X_test,
        }
        print(f"  [done] SHAP computed for {model_name}, top feature: "
              f"{X_train.columns[np.argmax(mean_abs_shap)]} "
              f"(mean |SHAP|={mean_abs_shap.max():.4f})")


Dataset: Sparkov

--- Base model: lgb (variant: SMOTETomek--MI_k15--Sparkov--20260628_232016) ---
  [done] SHAP computed for lgb, top feature: amt (mean |SHAP|=3.0344)

--- Base model: cat (variant: EditedNearestNeighbours--ANOVA_k30--Sparkov--20260629_035419) ---
  [done] SHAP computed for cat, top feature: hour (mean |SHAP|=0.7189)

--- Base model: isolation_forest (variant: EditedNearestNeighbours--ANOVA_Percentile10--Sparkov--20260629_122201) ---


  0%|          | 0/370479 [00:00<?, ?it/s]

  [done] SHAP computed for isolation_forest, top feature: card_amt_mean (mean |SHAP|=0.0604)

--- Base model: xgb (variant: TomekLinks--MI_k10--Sparkov--20260628_220840) ---
  [done] SHAP computed for xgb, top feature: amt (mean |SHAP|=2.6578)

--- Base model: sgd_log (variant: RandomUnderSampler--ANOVA_kall--Sparkov--20260629_072844) ---
  [skip] SHAP not supported for sgd_log

Dataset: EuropeanCard

--- Base model: xgb (variant: EditedNearestNeighbours--MI_k20--EuropeanCard--20260630_061718) ---
  [done] SHAP computed for xgb, top feature: V4 (mean |SHAP|=1.1792)

--- Base model: dl_cnn (variant: TomekLinks--ANOVA_k10--EuropeanCard--20260630_060828) ---
  [skip] Could not train model dl_cnn: Deep learning model 'dl_cnn' is not supported for SHAP retraining.

Dataset: IEEE-CIS

--- Base model: xgb (variant: EditedNearestNeighbours--ANOVA_kall--IEEE-CIS--20260630_015028) ---
  [done] SHAP computed for xgb, top feature: C5 (mean |SHAP|=0.4800)

--- Base model: isolation_forest (variant:

  0%|          | 0/118108 [00:00<?, ?it/s]

  [done] SHAP computed for isolation_forest, top feature: ProductCD (mean |SHAP|=0.0373)

--- Base model: sgd_huber (variant: BorderlineSMOTE--ANOVA_Percentile10--IEEE-CIS--20260630_024902) ---
  [skip] SHAP not supported for sgd_huber

--- Base model: gaussian_nb (variant: SMOTEENN--ANOVA_k5--IEEE-CIS--20260629_200424) ---


  0%|          | 0/118108 [00:00<?, ?it/s]

  [done] SHAP computed for gaussian_nb, top feature: V123 (mean |SHAP|=0.0826)


In [6]:
# =====================================================================
#  Meta-Learner Factory (same as in optimized_hybrid)
# =====================================================================
def get_meta_learners(random_state=42):
    """Return a dictionary of diverse meta-classifiers."""
    return {
        "LogisticRegression": LogisticRegression(
            C=1.0, max_iter=1000, random_state=random_state
        ),
        "RidgeClassifier": RidgeClassifier(
            alpha=1.0, random_state=random_state
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=200, max_depth=4, min_samples_leaf=10,
            random_state=random_state, n_jobs=-1
        ),
        "GaussianNB": GaussianNB(),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32), activation='relu',
            solver='adam', alpha=0.0001, max_iter=500,
            early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=10, random_state=random_state
        ),
        "LinearSVC": LinearSVC(
            C=1.0, max_iter=2000, random_state=random_state
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="auc", random_state=random_state,
            verbosity=0, use_label_encoder=False
        ),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=200, learning_rate=0.05,
            num_leaves=31, subsample=0.8, colsample_bytree=0.8,
            random_state=random_state, verbose=-1, n_jobs=-1
        ),
    }

In [7]:
# =====================================================================
#  Level 2: Meta‑Model SHAP Analysis
# =====================================================================
meta_shap_results = {}   # ds -> {meta_name, shap_values_meta, mean_abs_meta, base_model_names}

for ds, info in best_ensembles.items():
    print(f"\n--- Meta‑model SHAP for {ds} ---")
    meta_name = info["meta_learner"]

    # Load validation/test meta‑features for selected models
    # We saved them in optimized_hybrid folder
    out_dir = Path(CONFIG["OUTPUT_DIR"]).parent / "optimized_hybrid"
    val_meta_file = out_dir / f"{ds}_{info['strategy']}_selected_val_meta.csv"
    test_meta_file = out_dir / f"{ds}_{info['strategy']}_selected_test_meta.csv"
    if not val_meta_file.exists() or not test_meta_file.exists():
        print(f"  [skip] Meta‑feature files missing for {ds}")
        continue

    X_val_meta = pd.read_csv(val_meta_file)
    X_test_meta = pd.read_csv(test_meta_file)

    # Load true labels
    y_val = pd.read_csv(Path(CONFIG["DATA_ROOT"]) / f"{ds}_val.csv")[CONFIG["TARGET_COL"]].values
    y_test = pd.read_csv(Path(CONFIG["OUTPUT_DIR"]).parent / "optimized_hybrid" / f"{ds}_selected_test_labels.csv")["y_true"].values

    # Recreate meta-learner (same as in optimized_hybrid)
    from sklearn.base import clone
    meta_learners = get_meta_learners(CONFIG["RANDOM_STATE"])
    meta = clone(meta_learners[meta_name])
    meta.fit(X_val_meta, y_val)

    # SHAP for meta-model
    explainer = None
    if meta_name in ["LogisticRegression", "RidgeClassifier", "LinearSVC"]:
        explainer = shap.LinearExplainer(meta, X_val_meta)
    elif meta_name in ["RandomForest", "XGBoost", "LightGBM"]:
        explainer = shap.TreeExplainer(meta)
    else:
        try:
            explainer = shap.KernelExplainer(meta.predict_proba, shap.sample(X_val_meta, 50))
        except:
            print(f"  [skip] SHAP not supported for meta-learner {meta_name}")
            continue

    shap_values_meta = explainer.shap_values(X_test_meta)
    if isinstance(shap_values_meta, list):
        shap_values_meta = shap_values_meta[1]
    mean_abs_meta = np.abs(shap_values_meta).mean(axis=0)

    meta_shap_results[ds] = {
        "meta_name": meta_name,
        "shap_values_meta": shap_values_meta,
        "mean_abs_meta": mean_abs_meta,
        "base_model_names": X_test_meta.columns.tolist(),
    }
    print(f"  [done] Meta SHAP for {ds}, top base model: "
          f"{X_test_meta.columns[np.argmax(mean_abs_meta)]} "
          f"(mean |SHAP|={mean_abs_meta.max():.4f})")


--- Meta‑model SHAP for Sparkov ---


Background dataset has 56961 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=56961 when initializing the masker.


  [done] Meta SHAP for Sparkov, top base model: EditedNearestNeighbours--ANOVA_Percentil__isolation_forest (mean |SHAP|=0.0034)

--- Meta‑model SHAP for EuropeanCard ---
  [done] Meta SHAP for EuropeanCard, top base model: EditedNearestNeighbours--MI_k20--Europea__xgb (mean |SHAP|=0.0013)

--- Meta‑model SHAP for IEEE-CIS ---
  [done] Meta SHAP for IEEE-CIS, top base model: EditedNearestNeighbours--ANOVA_kall--IEE__xgb (mean |SHAP|=1.2754)


In [8]:
# =====================================================================
#  Level 3: Weighted Combined Importance (Robust)
# =====================================================================
# Φ_j = Σ_a φ_j^(a) * ψ_a
# where φ_j^(a) = mean|SHAP| of feature j for base model a (from Level 1)
#       ψ_a    = mean|SHAP| of base model a for meta-model (from Level 2)

combined_importance = {}

for ds in best_ensembles.keys():
    if ds not in shap_base_results or ds not in meta_shap_results:
        print(f"[skip] Missing SHAP results for {ds}")
        continue

    # Get meta mean|SHAP| for each base model
    meta_mean = np.asarray(meta_shap_results[ds]["mean_abs_meta"], dtype=float).flatten()
    base_names = meta_shap_results[ds]["base_model_names"]

    # Load selected models info to align meta columns with base models
    selected = selected_models_info[ds]["selected_models"]
    meta_col_names = []
    for sel in selected:
        variant_short = sel["variant"][:40]
        model = sel["model"]
        meta_col_names.append(f"{variant_short}__{model}")

    # Collect base SHAP vectors and their feature names
    phi_all = {}
    feature_sets = {}
    for idx, sel in enumerate(selected):
        model_key = sel["model"]
        if model_key not in shap_base_results[ds]:
            print(f"  [warn] Base SHAP missing for {model_key}, skipping in combined importance.")
            continue

        phi_a = np.asarray(shap_base_results[ds][model_key]["mean_abs_shap"], dtype=float).flatten()
        features_a = list(shap_base_results[ds][model_key]["feature_names"])

        if len(phi_a) != len(features_a):
            print(f"  [warn] Mismatch lengths for {model_key}: SHAP {len(phi_a)} vs features {len(features_a)}. Skipping.")
            continue

        phi_all[model_key] = phi_a
        feature_sets[model_key] = features_a

    if not phi_all:
        print(f"  [skip] No valid base SHAP for {ds}")
        continue

    # Build union of all features
    all_features = []
    for feats in feature_sets.values():
        for f in feats:
            if f not in all_features:
                all_features.append(f)
    feature_to_idx = {f: i for i, f in enumerate(all_features)}

    # Initialize combined importance vector
    phi_combined = np.zeros(len(all_features), dtype=float)

    # For each selected model, add weighted contribution
    for idx, sel in enumerate(selected):
        model_key = sel["model"]
        if model_key not in phi_all:
            continue

        phi_a = phi_all[model_key]
        features_a = feature_sets[model_key]

        # Determine meta weight for this base model
        # Try to find the corresponding column in meta SHAP
        try:
            meta_idx = base_names.index(meta_col_names[idx])
            weight = float(meta_mean[meta_idx])
        except (ValueError, IndexError):
            # Fallback: average meta weight
            weight = float(np.mean(meta_mean))

        # Add contribution for each feature
        for j, feat in enumerate(features_a):
            idx_global = feature_to_idx[feat]
            phi_combined[idx_global] += phi_a[j] * weight

    combined_importance[ds] = {
        "features": all_features,
        "combined_scores": phi_combined,
    }

    # Print top-5
    top_idx = np.argsort(phi_combined)[::-1][:5]
    print(f"\n[combined] {ds}: Top-5 combined important features:")
    for idx in top_idx:
        print(f"   {all_features[idx]}: {phi_combined[idx]:.4f}")

  [warn] Mismatch lengths for isolation_forest: SHAP 8 vs features 4. Skipping.
  [warn] Base SHAP missing for sgd_log, skipping in combined importance.

[combined] Sparkov: Top-5 combined important features:
   amt: 0.0197
   hour: 0.0092
   category: 0.0087
   log_amt: 0.0068
   card_amt_min: 0.0028
  [warn] Base SHAP missing for dl_cnn, skipping in combined importance.

[combined] EuropeanCard: Top-5 combined important features:
   V4: 0.0015
   Time: 0.0012
   V12: 0.0010
   V10: 0.0010
   V14: 0.0009
  [warn] Mismatch lengths for isolation_forest: SHAP 20 vs features 10. Skipping.
  [warn] Base SHAP missing for sgd_huber, skipping in combined importance.
  [warn] Mismatch lengths for gaussian_nb: SHAP 10 vs features 5. Skipping.

[combined] IEEE-CIS: Top-5 combined important features:
   C5: 0.6122
   C1: 0.3201
   C13: 0.3042
   TransactionDT: 0.2927
   C14: 0.2862


In [11]:
# =====================================================================
#  Level 4: LLM Explanation (Groq SDK)
# =====================================================================
import getpass
from groq import Groq

GROQ_API_KEY = getpass.getpass("GROQ_API_KEY: ")


if not GROQ_API_KEY:
    print("[warn] GROQ_API_KEY empty. Please set it to use Groq.")
else:
    client = Groq(api_key=GROQ_API_KEY)
    GROQ_MODEL = "openai/gpt-oss-120b"

    for ds, imp in combined_importance.items():
        features = imp["features"]
        scores = imp["combined_scores"]
        top_features = sorted(zip(features, scores), key=lambda x: -x[1])[:10]

        prompt = f"""
You are an AI assistant explaining fraud detection model decisions.
Dataset: {ds}
The model is an ensemble of several base models combined by a meta-learner.
Top contributing features (weighted combined importance):
{chr(10).join([f"- {name}: {score:.4f}" for name, score in top_features])}

Please provide a concise, non-technical explanation of why a transaction might be flagged as fraudulent based on these features. Include which features are most influential and why they matter.
"""

        try:
            chat_completion = client.chat.completions.create(
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt}
                ],
                model=GROQ_MODEL,
                temperature=0.5,
                max_completion_tokens=1024,
                top_p=1,
                stream=False,
            )
            explanation = chat_completion.choices[0].message.content
            print(f"\n[LLM] {ds} explanation:")
            print(explanation)

            # Save to file
            with open(Path(CONFIG["OUTPUT_DIR"]) / f"{ds}_llm_explanation.txt", "w") as f:
                f.write(explanation)
        except Exception as e:
            print(f"[error] LLM generation failed for {ds}: {e}")

GROQ_API_KEY:  ········



[LLM] Sparkov explanation:
**Why the system may flag a transaction as fraud**

The model looks at a handful of clues that together tell whether a purchase looks “out of the ordinary” for you and for the merchant. The three clues that matter most are:

| Feature (most to least influential) | What it tells the model |
|---|---|
| **amt** (transaction amount) | Very large or very small purchases are rarer, so an amount that is far from what you normally spend raises a red flag. |
| **hour** (time of day) | Buying at odd hours (e.g., late‑night or very early‑morning) is less common for most people, so the model is more suspicious of transactions that happen then. |
| **category** (type of purchase) | Some product categories are targeted more often by fraudsters. If you suddenly buy something in a high‑risk category (e.g., luxury goods, online gaming, etc.) the risk score goes up. |

The next set of clues fine‑tune the decision:

* **log_amt** – a transformed view of the amount that helps 

In [10]:
# =====================================================================
#  Save XAI Outputs
# =====================================================================
xai_summary = {}
for ds in combined_importance.keys():
    xai_summary[ds] = {
        "top_features": combined_importance[ds]["features"][:10],
        "combined_scores": combined_importance[ds]["combined_scores"][:10].tolist(),
        "meta_model": best_ensembles[ds]["meta_learner"],
        "base_models": [m for m in shap_base_results[ds].keys()],
    }

with open(Path(CONFIG["OUTPUT_DIR"]) / "xai_summary.json", "w") as f:
    json.dump(xai_summary, f, indent=2)

print(f"\n[final] XAI results saved to {CONFIG['OUTPUT_DIR']}")


[final] XAI results saved to ./trained/xai
